# VisionBridge — persistent hand-aware training (Lightning Studio)

This notebook is the persistent counterpart to `train_base_model_colab.ipynb`. It uses the repository's canonical hand-aware PyTorch trainer from a persistent Lightning Studio workspace. All four synchronized streams are first-class: pose 132, face 1404, left hand 63, right hand 63.

The notebook is a runner/orchestrator, not a second copy of the model implementation. That prevents the two training paths from quietly becoming different machines, which is how software grows a second personality.

In [ ]:
# 1. Runtime + persistent repository
import os,sys,subprocess
from pathlib import Path
import torch
WORKSPACE='/teamspace/studios/this_studio' if Path('/teamspace/studios/this_studio').is_dir() else str(Path.home())
REPO=Path(WORKSPACE)/'VisionBridge'
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/BharathWaj-K-R/VisionBridge.git',str(REPO)],check=True)
else:
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip()
    if dirty: raise RuntimeError('STOP: persistent workspace has local changes. Review them before pulling.')
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
os.chdir(REPO); sys.path.insert(0,str(REPO/'backend'))
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Repository:',REPO)
print('HEAD:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'],text=True).strip())
print('Device:',DEVICE)
if DEVICE=='cuda': print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
# 2. Verify the hand-aware repository contract
from app.models.base_model import POSE_INPUT_DIM, FACE_INPUT_DIM, HAND_INPUT_DIM
from app.training.isltranslate import ISLTranslateKeypointDataset, SimpleCharTokenizer
from app.models.base_model import VisionBridgeBaseModel
print('Pose:',POSE_INPUT_DIM,'Face:',FACE_INPUT_DIM,'Hand:',HAND_INPUT_DIM)
assert (POSE_INPUT_DIM,FACE_INPUT_DIM,HAND_INPUT_DIM)==(132,1404,63)
model=VisionBridgeBaseModel(vocab_size=49,use_hands=True)
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Total parameters:',sum(p.numel() for p in model.parameters()))
print('Trainable parameters:',trainable)
assert trainable>0
print('MODEL CONTRACT: PASS')


## Persistent data

Set `DATA_DIR` to the processed dataset directory on the persistent workspace. The directory must contain `ISLTranslate.csv`, `pose/`, `face/`, `left_hand/`, and `right_hand/`. If the data is not prepared yet, use the canonical Colab extraction notebook first, then point this notebook to the persistent copy.

In [ ]:
# 3. Validate persistent processed dataset
DATA_DIR=Path(os.environ.get('VB_DATA_DIR',str(REPO/'data/processed/isltranslate')))
tok=SimpleCharTokenizer()
required=[DATA_DIR/'ISLTranslate.csv',DATA_DIR/'pose',DATA_DIR/'face',DATA_DIR/'left_hand',DATA_DIR/'right_hand']
print('DATA_DIR:',DATA_DIR)
for p in required: print(p, 'FOUND' if p.exists() else 'MISSING')
if not all(p.exists() for p in required): raise FileNotFoundError('Persistent hand-aware dataset is incomplete.')
ds=ISLTranslateKeypointDataset(DATA_DIR,tokenizer=tok)
print('Examples:',len(ds),'Vocab:',tok.vocab_size)
for i in range(min(10,len(ds))):
    item=ds[i]
    assert item['pose'].shape[1]==132 and item['face'].shape[1]==1404
    assert item['left_hand'].shape[1]==63 and item['right_hand'].shape[1]==63
    assert item['pose'].shape[0]==item['face'].shape[0]==item['left_hand'].shape[0]==item['right_hand'].shape[0]
print('DATASET CONTRACT: PASS')


In [ ]:
# 4. Semantic overfit gate before expensive training
gate=subprocess.run([sys.executable,'-m','app.training.overfit_sanity','--data-dir',str(DATA_DIR),'--samples','2','--steps','2500','--lr','0.002'],cwd=REPO,env={**os.environ,'PYTHONPATH':str(REPO/'backend')},text=True,capture_output=True)
print(gate.stdout)
if gate.stderr: print('STDERR:
',gate.stderr)
if gate.returncode!=0: raise RuntimeError('SEMANTIC OVERFIT GATE FAILED — persistent full training is blocked.')
print('SEMANTIC OVERFIT GATE: PASS')


In [ ]:
# 5. Canonical repository trainer. Resumable checkpoints live outside the source tree.
OUTPUT=REPO/'backend/app/models/weights/base_model.pt'
CHECKPOINT_DIR=Path(WORKSPACE)/'visionbridge_training_checkpoints'
cmd=[sys.executable,'-m','app.training.train_base_model','--data-dir',str(DATA_DIR),'--output',str(OUTPUT),'--epochs',str(int(os.environ.get('VB_EPOCHS','20'))),'--batch-size',str(int(os.environ.get('VB_BATCH_SIZE','4'))),'--lr',str(float(os.environ.get('VB_LR','3e-4'))),'--weight-decay','1e-2','--max-grad-norm','1.0','--seed','42','--num-workers',str(int(os.environ.get('VB_NUM_WORKERS','4'))),'--checkpoint-dir',str(CHECKPOINT_DIR)]
print('TRAIN COMMAND:',' '.join(cmd))
result=subprocess.run(cmd,cwd=REPO,env={**os.environ,'PYTHONPATH':str(REPO/'backend')})
if result.returncode!=0: raise RuntimeError('TRAINING FAILED')
VOCAB=OUTPUT.with_suffix('.vocab.json')
if not OUTPUT.exists() or not VOCAB.exists(): raise RuntimeError('Training ended without checkpoint + vocabulary.')
print('TRAINING: PASS')


In [ ]:
# 6. Resume instructions — this cell only reports state and does not mutate it
latest=CHECKPOINT_DIR/'latest.pt'
print('Latest resumable checkpoint:',latest)
print('Exists:',latest.exists())
print('To resume after an interruption, rerun Step 5 with VB_EPOCHS higher and add --resume to the command.')
